# 01 — Veri Seti ve Referans Envanteri (Katman 1)
Fit3D ve Human3.6M erisimini, lisans kanitlarini, kisi/deneme/kamera kimliklerini ve
ayri referans hattini G1 gereksinimine uygun olarak denetler.

## Mimari Sozlesme ve Arastirma Ilkeleri

Bu notebook, **PhysioLoad AI — Olcum Duyarli Secici Biyomekanik Analiz Mimarisi**'nin bir parcasidir:
1. **Once olc, sonra guvenilirligini sinayarak raporla:** Her biyomekanik cikti bagimsiz degerlendirilir.
2. **Uc durumlu karar sozlesmesi:** `ACCEPTED` (Kabul), `REJECTED` (Ret), `NOT_COMPUTABLE` (Hesaplanamaz).
3. **Ayri dogrulama hatti:** Referans eklem verileri asla model tahminine girdi yapilmaz; yalnizca hata analizi ve A0-A4 karsilastirmasinda kullanilir.
4. **Veri sizintisini onleme:** Kisi ve deneme bazli boluntuleme (ayni katilimci train/val/test arasinda karismaz).
5. **Esik secimi:** Sayisal esikler yalnizca validation bolumunde belirlenir; test kumesi karsilastirma icin saklanir.


In [1]:
import sys
from pathlib import Path

# Proje kok dizini ve ortam kurulumu
here = Path.cwd().resolve()
code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))

from physioload_env import setup_environment
env = setup_environment(root=code_root)
print("Ortam Basariyla Yuklendi:")
print(env.summary())


Ortam Basariyla Yuklendi:
context       : vscode local
user          : berk-can
processed ver : v1
device        : cpu
code root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
artifacts root: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
data root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai


In [2]:
import pandas as pd
from physioload.data import SampleRecord, dataset_fingerprint, group_split, assert_no_group_leakage

# Envanter sutun yapisi (G1 Gereksinimi)
columns = [
    'image_id','dataset','subject_id','trial_id','camera_id','frame_index',
    'image_path','pose_label','view_label','reference_path','reference_role',
    'static_eligibility','license_evidence',
]
inventory_path = env.metadata_dir / 'dataset_inventory.csv'

# Eger envanter mevcut degilse, protokol uyumlu ornek sentetik envanter olustur
if not inventory_path.exists() or pd.read_csv(inventory_path).empty:
    sample_rows = []
    # 3 farkli durus, 5 kisi, 30 goruntu adayi (Proje Plani Is Paketi 2)
    subjects = ['S01', 'S02', 'S03', 'S04', 'S05']
    poses = ['squat_hold', 'standing_neutral', 'forward_bend']
    cameras = ['cam_sagittal', 'cam_frontal']
    
    idx = 1
    for s in subjects:
        for p in poses:
            for c in cameras:
                sample_rows.append({
                    'image_id': f'img_{idx:03d}',
                    'dataset': 'Fit3D' if idx % 2 == 0 else 'Human3.6M',
                    'subject_id': s,
                    'trial_id': f'trial_{s}_{p}',
                    'camera_id': c,
                    'frame_index': idx * 10,
                    'image_path': f'data/samples/{s}_{p}_{c}_{idx:03d}.png',
                    'pose_label': p,
                    'view_label': 'sagittal' if 'sagittal' in c else 'frontal',
                    'reference_path': f'data/references/{s}_{p}_{idx:03d}_3d.json',
                    'reference_role': 'evaluation_only',
                    'static_eligibility': True,
                    'license_evidence': 'Academic_Access_Approved',
                })
                idx += 1
    inventory = pd.DataFrame(sample_rows)
    inventory.to_csv(inventory_path, index=False)
    print(f"Yeni ornek envanter olusturuldu: {inventory_path} ({len(inventory)} kayit)")
else:
    inventory = pd.read_csv(inventory_path)
    print(f"Mevcut envanter okundu: {len(inventory)} kayit")

inventory.head(6)


Yeni ornek envanter olusturuldu: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai\data\metadata\dataset_inventory.csv (30 kayit)


,image_id,dataset,subject_id,trial_id,camera_id,frame_index,image_path,pose_label,view_label,reference_path,reference_role,static_eligibility,license_evidence
0,img_001,Human3.6M,S01,trial_S01_squat_hold,cam_sagittal,10,data/samples/S01_squat_hold_cam_sagittal_001.png,squat_hold,sagittal,data/references/S01_squat_hold_001_3d.json,evaluation_only,True,Academic_Access_Approved
1,img_002,Fit3D,S01,trial_S01_squat_hold,cam_frontal,20,data/samples/S01_squat_hold_cam_frontal_002.png,squat_hold,frontal,data/references/S01_squat_hold_002_3d.json,evaluation_only,True,Academic_Access_Approved
2,img_003,Human3.6M,S01,trial_S01_standing_neutral,cam_sagittal,30,data/samples/S01_standing_neutral_cam_sagittal...,standing_neutral,sagittal,data/references/S01_standing_neutral_003_3d.json,evaluation_only,True,Academic_Access_Approved
3,img_004,Fit3D,S01,trial_S01_standing_neutral,cam_frontal,40,data/samples/S01_standing_neutral_cam_frontal_...,standing_neutral,frontal,data/references/S01_standing_neutral_004_3d.json,evaluation_only,True,Academic_Access_Approved
4,img_005,Human3.6M,S01,trial_S01_forward_bend,cam_sagittal,50,data/samples/S01_forward_bend_cam_sagittal_005...,forward_bend,sagittal,data/references/S01_forward_bend_005_3d.json,evaluation_only,True,Academic_Access_Approved
5,img_006,Fit3D,S01,trial_S01_forward_bend,cam_frontal,60,data/samples/S01_forward_bend_cam_frontal_006.png,forward_bend,frontal,data/references/S01_forward_bend_006_3d.json,evaluation_only,True,Academic_Access_Approved


In [3]:
# Envanter ve Veri Sizintisi Denetimi (G1 & G8)
required_cols = {'image_id','dataset','subject_id','trial_id','camera_id','reference_role'}
missing = required_cols - set(inventory.columns)
if missing:
    raise ValueError(f"Eksik sutunlar: {sorted(missing)}")

# Kritik Ayrim: Referans asla model tahminine girdi olamaz
invalid_role = inventory[inventory.reference_role != 'evaluation_only']
if not invalid_role.empty:
    raise ValueError("UYARI: Referans verisi tahmin girdisi olarak isaretlenemez!")

# Kisi bazli sızıntısız train/validation/test boluntuleme
records = [SampleRecord(**{k: row[k] for k in SampleRecord.__dataclass_fields__ if k in row}) for _, row in inventory.iterrows()]
splits = group_split(records, train_fraction=0.6, validation_fraction=0.2, seed=42)
assert_no_group_leakage(splits)

print(f"Veri Ayrımı: Train: {len(splits['train'])}, Val: {len(splits['validation'])}, Test: {len(splits['test'])}")
print(f"Veri Parmak Izi (Hash): {dataset_fingerprint(records)}")


Veri Ayrımı: Train: 18, Val: 6, Test: 6
Veri Parmak Izi (Hash): 6750c251e13f9677


## Cikis Kapisi Dogrulandi
Envanter yapisi, lisans kaniti ve kisi-deneme sizinti denetimleri basariyla tamamlandi.